# Notebook 2: Train a Model and search with it

```
data/crm.csv  →  [1. Prepare data]  →  [2. Train & search]
                                        you are here
```

**Goal:** search customers in ways a database filter cannot.

| Question | Database / filter | Vector search |
|---|---|---|
| *"unhappy healthcare customers"* | You must know the columns are `industry` and `satisfaction_score`, and choose `< 5` yourself | Type the words |
| *"customers like CUST-00042"* | There is no "like" in SQL | Find the nearest vectors |
| A new lead where we only know 3 fields | You must write a new filter by hand | Encode what you know and find the nearest vectors |
| Which results are best? | A row either matches or not | Every customer gets a score |

**The idea: an Encoder–Decoder (autoencoder)**

```
                 ┌─────────┐                ┌─────────┐
customer row ──► │ Encoder │ ──► vector ──► │ Decoder │ ──► customer row (rebuilt)
 (26 fields)     └─────────┘  (32 numbers)  └─────────┘      (26 fields)
```

If the Decoder can rebuild a customer from only 32 numbers, then those 32 numbers are a good **summary** of the customer.

Similar customers get similar summaries, so we can search by comparing vectors.

In [ ]:
import random
import torch
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

print(f"Available Device: {device}")

## Load the prepared data

These are the files saved by notebook 1.

In [ ]:
import json
import pandas as pd


data  = np.load("outputs/prepared/data.npz")
stats = json.loads(open("outputs/prepared/stats.json").read())
tags  = json.loads(open("outputs/prepared/tags.json").read())
df    = pd.read_csv("data/crm.csv", dtype=str, keep_default_na=False)

X = { k: torch.tensor(data[k]).to(device) for k in ["cat", "num", "text", "missing"] }

is_test = data["is_test"]
train_rows = np.flatnonzero(~is_test)


print({k: tuple(v.shape) for k, v in X.items()})

### One extra field: `description`

To search with English we need a way to give the model a query. We add a extra field called `description`.

* For real customers, `description` is empty (marked *missing*).
* A **search query is a customer where we only know the description**. All the other fields are missing.

So customers and queries go through the **same Encoder** and end up in the same space.

In [ ]:
CAT_COLS = list(stats["vocab"])                                    # 12 category columns
NUM_COLS = list(stats["mean"])                                     # 10 number and date columns
TEXT_COLS = ["notes", "interests", "last_ticket_subject", "description"]
N_FIELDS = len(CAT_COLS) + len(NUM_COLS) + len(TEXT_COLS)          # 26 fields
LOG_COLS = ["total_orders", "monthly_recurring_revenue", "lifetime_value", "avg_order_value"]

# add the empty description field to every customer
X["text"] = torch.cat([X["text"], torch.zeros_like(X["text"][:, :1])], dim=1)
X["missing"] = torch.cat([X["missing"], torch.ones_like(X["missing"][:, :1])], dim=1)
print({k: tuple(v.shape) for k, v in X.items()})

`text_to_ids` is copied from notebook 1. `form` turns a few known fields into model input; every other field is marked missing.

In [ ]:
# X.items()

In [ ]:
import re
import zlib


BUCKETS = 50_000
TEXT_LEN = 100


def text_to_ids(text, max_len=TEXT_LEN):
    words = re.findall(r"[a-z0-9]+", text.lower())
    words = words + [a + "_" + b for a, b in zip(words, words[1:])]
    ids = [1 + zlib.crc32(w.encode()) % (BUCKETS - 1) for w in words][:max_len]
    return ids + [0] * (max_len - len(ids))


def customers(rows):
    return {k: v[rows] for k, v in X.items()}


def form(*known_fields):
    """One input row per dict, e.g. form({"description": "unhappy"}, {"industry": "Retail", "satisfaction_score": 3})"""
    n = len(known_fields)

    cat = np.zeros((n, len(CAT_COLS)), dtype=int)
    num = np.zeros((n, len(NUM_COLS)), dtype="float32")
    text = np.zeros((n, len(TEXT_COLS), TEXT_LEN), dtype=int)
    missing = np.ones((n, N_FIELDS), dtype=bool)

    for i, fields in enumerate(known_fields):
        for col, value in fields.items():
            if col in CAT_COLS:
                j = CAT_COLS.index(col)
                cat[i, j] = stats["vocab"][col].get(str(value), 0)
                missing[i, j] = cat[i, j] == 0
            elif col in NUM_COLS:
                j = NUM_COLS.index(col)
                value = np.log1p(float(value)) if col in LOG_COLS else float(value)
                num[i, j] = (value - stats["mean"][col]) / stats["std"][col]
                missing[i, len(CAT_COLS) + j] = False
            else:
                j = TEXT_COLS.index(col)
                text[i, j] = text_to_ids(value)
                missing[i, len(CAT_COLS) + len(NUM_COLS) + j] = False

    batch = {"cat": cat, "num": num, "text": text, "missing": missing}

    return {k: torch.tensor(v, device=device) for k, v in batch.items()}


form({"description": "unhappy healthcare customers"})["missing"].int()

## Encoder / Decoder Design

### Encoder

```
 city  industry … plan   orders  revenue …  notes  interests  ticket  description
   │      │        │       │       │          │        │         │         │
  64     64       64      64      64         64       64        64        64    ← a vector per field
   └─────────────────── put them side by side: 26 × 64 = 1664 numbers ─────────┘     (missing field = zeros)
                                          │
                                 MLP: 1664 → 256 → 32
                                          │
                                  customer vector
```

Each field becomes a vector of 64 numbers:

* category → a lookup table (`nn.Embedding`)
* number → `value × learned vector + learned vector`
* text → the average of its word vectors (`nn.EmbeddingBag`)

The final vector is scaled to length 1, so **similarity = dot product**.

In [ ]:
from torch import nn
import torch.nn.functional as F


FIELD_DIM = 64   # numbers per field
DIM = 32         # numbers per customer vector


class Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.cat_emb = nn.ModuleList([nn.Embedding(len(stats["vocab"][c]) + 1, FIELD_DIM) for c in CAT_COLS])
        self.num_weight = nn.Parameter(torch.randn(len(NUM_COLS), FIELD_DIM))
        self.num_bias = nn.Parameter(torch.randn(len(NUM_COLS), FIELD_DIM))
        self.word_emb = nn.EmbeddingBag(BUCKETS, FIELD_DIM, mode="mean", padding_idx=0)

        self.mlp = nn.Sequential(
            nn.Linear(N_FIELDS * FIELD_DIM, 256),
            nn.ReLU(),
            nn.Linear(256, DIM)
        )


    def forward(self, cat, num, text, missing):
        cat_fields = torch.stack([emb(cat[:, i]) for i, emb in enumerate(self.cat_emb)], dim=1)  # [B, 12, 64]
        num_fields = num.unsqueeze(-1) * self.num_weight + self.num_bias                         # [B, 10, 64]
        text_fields = self.word_emb(text.flatten(0, 1)).view(len(text), -1, FIELD_DIM)           # [B, 4, 64]

        fields = torch.cat([cat_fields, num_fields, text_fields], dim=1)                         # [B, 26, 64]
        fields = fields * ~missing.unsqueeze(-1)                                                 # missing -> 0

        return F.normalize(self.mlp(fields.flatten(1)), dim=-1)                                  # [B, 32]

### Decoder

The Decoder goes the other way. From the vector, it guesses every category (a classifier per column) and every number.

We don't rebuild the text fields. The text is only used as **input**.

In [ ]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(DIM, 256),
            nn.ReLU()
        )

        self.cat_heads = nn.ModuleList([nn.Linear(256, len(stats["vocab"][c]) + 1) for c in CAT_COLS])
        self.num_head = nn.Linear(256, len(NUM_COLS))


    def forward(self, z):
        h = self.mlp(z)
        return [head(h) for head in self.cat_heads], self.num_head(h)

Build Encoder and Decoder

In [ ]:
encoder = Encoder().to(device)
decoder = Decoder().to(device)

z = encoder(**customers(torch.arange(4, device=device)))
cat_logits, num_pred = decoder(z)

print("4 customers   ->", tuple(z.shape))
print("rebuilt       ->", len(cat_logits), "category columns,", tuple(num_pred.shape), "numbers")
print("parameters    :", sum(p.numel() for p in [*encoder.parameters(), *decoder.parameters()]))

## Training Objective

### Game 1: fill in the blanks

If the Encoder sees every field, rebuilding them is too easy.

So we **hide half of the fields** and ask the Decoder to rebuild **all** of them.
To do this, the model has to learn how the fields relate to each other. 

For example, angry notes usually go with a low satisfaction score.

* categories: cross-entropy (did it pick the right value?)
* numbers: mean squared error (how far off is it?)


This is called a **denoising autoencoder**.

In [ ]:
HIDE = 0.5  # hide 50% of the fields


def hide_fields(batch):
    hidden = torch.rand(batch["missing"].shape, device=device) < HIDE
    return {**batch, "missing": batch["missing"] | hidden}


def reconstruction_loss(batch, cat_logits, num_pred):
    cat_loss = sum(F.cross_entropy(logits, batch["cat"][:, i], ignore_index=0)  # id 0 = unknown, skip it
                   for i, logits in enumerate(cat_logits)) / len(CAT_COLS)
    known = ~batch["missing"][:, len(CAT_COLS):len(CAT_COLS) + len(NUM_COLS)]   # only score real numbers
    num_loss = ((num_pred - batch["num"]) ** 2)[known].mean()
    return cat_loss + num_loss

### Game 2: match the description

Game 1 alone never teaches the model that *"unhappy"* means a low `satisfaction_score`. That word is not in any column.

So we use the **tags** from notebook 1 to make up descriptions such as *"frustrated healthcare company"*:

In [ ]:
def make_query(row):
    picked = random.sample(list(tags[row].items()), k=random.randint(1, 3))
    text = " ".join(random.choice(ways_to_say_it) for _, ways_to_say_it in picked)
    return text, [tag for tag, _ in picked]


for _ in range(5):
    print(make_query(0))

For a batch of customers we make one description each. Each description should be **closest to its own customer**
among everyone in the batch. This is a classification over the batch, so we use cross-entropy.

In [ ]:
TEMPERATURE = 0.1  # smaller = sharper matching


def match_loss(rows, customer_vectors):
    query_vectors = encoder(**form(*[{"description": make_query(r)[0]} for r in rows.tolist()]))
    similarity = query_vectors @ customer_vectors.T / TEMPERATURE  # [batch, batch]
    correct = torch.arange(len(rows), device=device)               # query i belongs to customer i
    return F.cross_entropy(similarity, correct)

## Model Train

Each step plays both games on a batch of training customers, adds the two losses and updates the weights.
It takes about 30 seconds on a GPU.

In [ ]:
EPOCHS = 200 if device == "cuda" else 40
BATCH  = 256
LR     = 1e-3

optimizer = torch.optim.Adam([*encoder.parameters(), *decoder.parameters()], lr=LR)
history = []

for epoch in range(1, EPOCHS + 1):
    encoder.train()
    decoder.train()

    np.random.shuffle(train_rows)

    epoch_losses = []

    for start in range(0, len(train_rows), BATCH):
        rows = torch.tensor(train_rows[start:start + BATCH], device=device)
        batch = customers(rows)

        loss1 = reconstruction_loss(batch, *decoder(encoder(**hide_fields(batch))))  # Game 1
        loss2 = match_loss(rows, encoder(**batch))                                   # Game 2
        loss = loss1 + loss2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_losses.append([loss1.item(), loss2.item()])

    rebuild, match = np.mean(epoch_losses, axis=0)
    history.append({"fill in the blanks": rebuild, "match the description": match})
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d}   rebuild loss {rebuild:.3f}   match loss {match:.3f}")

In [ ]:
from matplotlib import pyplot as plt


pd.DataFrame(history).plot(title="training loss", xlabel="epoch", figsize=(7, 3))
plt.show()

## Save the trained model

We save only the weights (`state_dict`). The model code stays in this notebook.

In [ ]:
torch.save(
    {
        "encoder": encoder.state_dict(),
        "decoder": decoder.state_dict()
    },
    "outputs/model.pt"
)
print("saved outputs/model.pt")

## Load the saved model

From here on we use the **loaded** model, so everything below also works without retraining:
skip the training cells and run this one.

In [ ]:
checkpoint = torch.load("outputs/model.pt", map_location=device)

encoder = Encoder().to(device)
decoder = Decoder().to(device)

encoder.load_state_dict(checkpoint["encoder"])
decoder.load_state_dict(checkpoint["decoder"])

encoder.eval()
decoder.eval()

print("loaded outputs/model.pt")

## Look inside the model

### The Decoder rebuilds a customer it has never seen

We take a **test** customer, hide half of its fields, encode it to 32 numbers and decode it again.

In [ ]:
encoder.eval()
decoder.eval()
id_to_value = {c: {i: v for v, i in stats["vocab"][c].items()} for c in CAT_COLS}


@torch.no_grad()
def decode(vector):
    cat_logits, num_pred = decoder(vector[None])
    out = {c: id_to_value[c].get(logits.argmax().item(), "?") for c, logits in zip(CAT_COLS, cat_logits)}
    for j, col in enumerate(NUM_COLS):
        value = num_pred[0, j].item() * stats["std"][col] + stats["mean"][col]
        out[col] = np.expm1(value) if col in LOG_COLS else value
    return out


SHOW_COLS = ["industry", "plan_tier", "lifecycle_stage", "country", "last_ticket_category",
             "satisfaction_score", "churn_risk", "lifetime_value", "total_orders"]

row = int(np.flatnonzero(is_test)[0])
torch.manual_seed(1)
half_hidden = hide_fields(customers(torch.tensor([row], device=device)))
rebuilt = decode(encoder(**half_hidden)[0])

hidden = half_hidden["missing"][0].cpu().numpy()
pd.DataFrame({
    "real": df.loc[row, SHOW_COLS],
    "hidden from the encoder": ["yes" if hidden[(CAT_COLS + NUM_COLS).index(c)] else "" for c in SHOW_COLS],
    "rebuilt": [rebuilt[c] if isinstance(rebuilt[c], str) else round(rebuilt[c], 2) for c in SHOW_COLS],
})

### How well is each category rebuilt? (all test customers, half the fields hidden)

In [ ]:
with torch.no_grad():
    test = customers(torch.tensor(np.flatnonzero(is_test), device=device))
    cat_logits, _ = decoder(encoder(**hide_fields(test)))
accuracy = {c: (logits.argmax(1) == test["cat"][:, i]).float().mean().item() for i, (c, logits) in enumerate(zip(CAT_COLS, cat_logits))}
pd.Series(accuracy).sort_values().plot.barh(title="rebuild accuracy", xlim=(0, 1), figsize=(6, 3.5))
plt.show()

The fields that searches ask about, such as industry, plan, stage and ticket, are rebuilt well. Fields that no search
asks about, such as `job_title`, are rebuilt worse. Game 2 told the 32 numbers what is **worth remembering**.

### What does the model imagine for a description?

We encode a description and **decode** it. This shows the "typical customer" the model has in mind for those words.

In [ ]:
@torch.no_grad()
def imagine(description):
    customer = decode(encoder(**form({"description": description}))[0])
    return {c: customer[c] if isinstance(customer[c], str) else round(customer[c], 2) for c in SHOW_COLS}


pd.DataFrame({q: imagine(q) for q in ["unhappy healthcare customers",
                                      "happy customers on enterprise plan",
                                      "likely to churn"]})

### Visualization of the customer space

We squeeze the 32 numbers down to 2 with t-SNE. Each dot is a customer. The model was never told to group customers by
industry, but every industry forms its own island. Lifecycle stage is mixed inside each island: the vector still holds
it (the Decoder rebuilds it well), but industry decides the overall layout.

In [ ]:
from sklearn.manifold import TSNE

with torch.no_grad():
    vectors = encoder(**customers(torch.arange(len(df), device=device))).cpu().numpy()

xy = TSNE(2, random_state=0).fit_transform(vectors)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col in zip(axes, ["industry", "lifecycle_stage"]):
    for value in sorted(df[col].unique()):
        m = (df[col] == value).to_numpy()
        ax.scatter(xy[m, 0], xy[m, 1], s=4, label=value)
    ax.set_title(col)
    ax.legend(markerscale=4, fontsize=8)
plt.show()

# Search with plain English

1. Encode every customer once and keep the vectors. This is the **vector index**.
2. Encode the query as a customer that only has a description.
3. Score every customer with `customer_vectors @ query_vector` and show the highest scores.

In [ ]:
with torch.no_grad():
    customer_vectors = encoder(**customers(torch.arange(len(df), device=device))).cpu().numpy()
print("vector index:", customer_vectors.shape)

SHOW = ["customer_id", "industry", "plan_tier", "lifecycle_stage", "satisfaction_score", "churn_risk",
        "lifetime_value", "city"]


def show(rows, scores):
    return df.iloc[rows][SHOW].assign(score=scores.round(3))


@torch.no_grad()
def search(query, k=8, where=None):
    q = encoder(**form({"description": query}))[0].cpu().numpy()
    scores = customer_vectors @ q
    if where is not None:
        scores[~where.to_numpy()] = -1  # an exact filter can still be added on top
    top = np.argsort(-scores)[:k]
    return show(top, scores[top])


search("unhappy healthcare customers")

None of these rows contains the word *"unhappy"*. Here is the database version of the same question. You have to know
the column names and choose the threshold yourself:

In [ ]:
satisfaction = pd.to_numeric(df.satisfaction_score)
df[(df.industry == "Healthcare") & (satisfaction < 5)][SHOW].head(8)

In [ ]:
search("high value enterprise accounts likely to churn")

In [ ]:
search("happy customers in Tokyo interested in fleet tracking")

Search and filter work together. Use a filter for **exact** rules (only the USA) and vector search for **fuzzy** ones:

In [ ]:
search("high value enterprise accounts likely to churn", where=df.country == "USA")

### Customers similar to one customer

No query text here: one customer's own vector is the query.

In [ ]:
def similar_to(customer_id, k=8):
    i = df.index[df.customer_id == customer_id][0]
    scores = customer_vectors @ customer_vectors[i]
    top = np.argsort(-scores)[1:k + 1]  # [0] is the customer itself
    return show(top, scores[top])


display(df[df.customer_id == "CUST-00042"][SHOW])
similar_to("CUST-00042")

### A brand-new customer

The model also works for customers it has never seen, with no retraining. We know only **4 of the 26 fields**.
The rest are missing, just like in Game 1.

In [ ]:
new_lead = form({
    "industry": "Hospitality",
    "plan_tier": "Enterprise",
    "satisfaction_score": 2,
    "notes": "Frustrated by repeated outages.",
})
with torch.no_grad():
    vector = encoder(**new_lead)[0].cpu().numpy()

scores = customer_vectors @ vector
top = np.argsort(-scores)[:8]
show(top, scores[top])

## Evaluation: `How good is it?`

We compare with **keyword search (TF-IDF)**, which turns each row into text (`"industry: Healthcare | plan_tier: Free | …"`)
and matches words.

We make 300 queries from **test** customers. For each query, we count how many of the top 10 results really have
all of the query's tags (precision@10).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

columns = CAT_COLS + ["total_orders", "lifetime_value", "satisfaction_score", "churn_risk", "notes", "interests"]
documents = df.apply(lambda r: " | ".join(f"{c}: {r[c]}" for c in columns), axis=1)
tfidf = TfidfVectorizer()
keyword_index = tfidf.fit_transform(documents)


def precision_at_10(scores, queries):
    return np.mean([np.mean([set(q_tags) <= set(tags[i]) for i in np.argsort(-s)[:10]])
                    for s, (_, q_tags) in zip(scores, queries)])


random.seed(1)
queries = [make_query(row) for row in random.choices(np.flatnonzero(is_test).tolist(), k=300)]
texts = [q for q, _ in queries]
with torch.no_grad():
    query_vectors = encoder(**form(*[{"description": t} for t in texts])).cpu().numpy()

print("encoder-decoder:", round(precision_at_10(query_vectors @ customer_vectors.T, queries), 3))
print("keyword search :", round(precision_at_10((tfidf.transform(texts) @ keyword_index.T).toarray(), queries), 3))

Keyword search cannot know that *"unhappy"* means a low `satisfaction_score`, or that *"likely to churn"* means a high
`churn_risk`. The model learned these links in Game 2.

## Save the vector index

The model was saved right after training. Here we save the customer vectors.

In [ ]:
np.save("outputs/customer_vectors.npy", customer_vectors)
print("saved outputs/customer_vectors.npy")

## Recap

* **Encoder:** 26 fields → 32 numbers. **Decoder:** 32 numbers → 26 fields.
* **Game 1, fill in the blanks:** hide half of the fields and rebuild all of them. The vector learns to summarize the customer.
* **Game 2, match the description:** a description should land next to its customer. The vector learns English words
  such as *unhappy* and *likely to churn*.
* **Search = one dot product** between the query vector and every customer vector. The same vectors give look-alikes,
  and they work for new customers with only a few known fields.

### ✏️ Try it

1. Train with only Game 1 (`loss = loss1`). Does `search(...)` still work? Does `similar_to(...)` still work? Why?
2. Set `HIDE = 0.0`. Then try the brand-new customer again.
3. Set `DIM = 2`. Now you can plot the vectors directly, without t-SNE. How much worse is the search?
4. Write a query the model gets wrong. (Hint: is it covered by the tags in notebook 1?)